# decrement_1d: proving termination with the DSL

Model the program as a `sugar.Module`, load its ranking function `V`, and verify with Z3 that
`V >= 0` and `V` strictly decreases on every loop iteration.

## The program

```c
int main() {
    int x = __VERIFIER_nondet_int();
    while (x > 0) {
        x = x - 1;
    }
    return 0;
}
```

The simplest terminating loop: decrement `x` until `x <= 0`. `x` is modelled over the
integers (`LIA`).

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import z3

from zrth import LRA, Var, Real, X, Bool
from zrth.sugar import Module, ite
from zrth.sugar import X as nxt
from zrth.torch import Module as RankingModule

## The program as a `sugar.Module`

`x` is a variable (`Var`), with nondeterministic initial value from an external input `x0`:
`init` awaits the external's next value (`nxt(x0)`, sugar's `X` aliased for readability). The
loop body is `x' = ite(x > 0, x - 1, x)`. `init` receives the external variables and `update`
the controlled-then-external variables, all as unpacked parameters.

In [ ]:
class Decrement(Module):
    def init(self, x0):
        return nxt(x0)                     # nondeterministic initial value

    def update(self, x, x0):
        return ite(x > 0, x - 1, x)        # while x > 0: x = x - 1


Real1 = Real([1, 1])
x = Var(Real1)
x0 = Var(Real1)
program = Decrement(theory=LRA, ctrl=(x,), extl=(x0,))
print(program.with_varnames({x : "x", x0 : "x0"}))

module
  external
    x0 : Real([1,1])
  interface
    x : Real([1,1])
  atom controls x reads x awaits x0
  init
    X(x) := Id X(x0)
  delay
    d(x) := ZERO 
  update
    #6 := [[0]] 
    #7 := Gt (x, #6)
    #8 := [[1]] 
    #9 := Sub (x, #8)
    #10 := Ite (#7, #9, x)
    X(x) := Id #10



`x` is controlled and `x0` is external, so the module is open; `update` is the loop
transition.

## The ranking function

Rebuild `V` from the JSON as an `nn.Module` and wrap it with `zrth.torch.Module`.

In [ ]:
# resolve the JSON whether cwd is this folder or the repo root
_name = "decrement_1d.ranking_function.json"
rf_path = next(d / _name for d in (Path.cwd(), Path.cwd() / "tutorials" / "decrement_1d") if (d / _name).exists())
rf = json.loads(rf_path.read_text())
layers = rf["layers"]
delta = rf["delta_threshold"]


class Ranking(nn.Module):
    """Linear -> ReLU -> Linear, with shapes and weights from the JSON."""

    def __init__(self, layers):
        super().__init__()
        (o1, i1), (o2, i2) = [(len(L["W"]), len(L["W"][0])) for L in layers]
        self.fc1 = nn.Linear(i1, o1)
        self.fc2 = nn.Linear(i2, o2)
        with torch.no_grad():
            self.fc1.weight.copy_(torch.tensor(layers[0]["W"]))
            self.fc1.bias.copy_(torch.tensor(layers[0]["b"]))
            self.fc2.weight.copy_(torch.tensor(layers[1]["W"]))
            self.fc2.bias.copy_(torch.tensor(layers[1]["b"]))

    def forward(self, s):
        return self.fc2(torch.relu(self.fc1(s)))

Real1 = Real([1,1])
PostV = Var(Real1)
Post = RankingModule(Ranking(layers), ctrl=PostV, extl=x, theory=LRA, combinatorial=True)
print(Post.with_varnames({x : "x", PostV : "PostV"}))

module
  external
    x : Real([1,1])
  interface
    PostV : Real([1,1])
  atom controls PostV awaits x
  init
    #17 := Transpose X(x)
    #18 := Linear([[0], [1], [2], [1], [-1], [1], [1]], [[-0], [-1], [0], [-2], [-1], [-1], [1]]) #17
    #19 := Transpose #18
    #20 := ReLU #19
    #21 := Transpose #20
    #22 := Linear([[2, 2, 2, 2, 2, 2, 2]], [[0]]) #21
    #23 := Transpose #22
    X(PostV) := Id #23
  delay
    d(PostV) := ZERO 
  update
    #17 := Transpose X(x)
    #18 := Linear([[0], [1], [2], [1], [-1], [1], [1]], [[-0], [-1], [0], [-2], [-1], [-1], [1]]) #17
    #19 := Transpose #18
    #20 := ReLU #19
    #21 := Transpose #20
    #22 := Linear([[2, 2, 2, 2, 2, 2, 2]], [[0]]) #21
    #23 := Transpose #22
    X(PostV) := Id #23



## Composition



In [ ]:
V = Var(Real1)
Pre = RankingModule(Ranking(layers), ctrl=V, extl=x, theory=LRA)

M = (program * Pre * Post).hide({x})
print(M.with_varnames({x : "x", x0 : "x0", PostV : "PostV", V: "V"}))

module
  external
    x0 : Real([1,1])
  interface
    PostV : Real([1,1])
    V : Real([1,1])
  private
    x : Real([1,1])
  atom controls x reads x awaits x0
  init
    X(x) := Id X(x0)
  delay
    d(x) := ZERO 
  update
    #6 := [[0]] 
    #7 := Gt (x, #6)
    #8 := [[1]] 
    #9 := Sub (x, #8)
    #10 := Ite (#7, #9, x)
    X(x) := Id #10
  atom controls V reads x
  init
    X(V) := HAVOC 
  delay
    d(V) := ZERO 
  update
    #30 := Transpose x
    #31 := Linear([[0], [1], [2], [1], [-1], [1], [1]], [[-0], [-1], [0], [-2], [-1], [-1], [1]]) #30
    #32 := Transpose #31
    #33 := ReLU #32
    #34 := Transpose #33
    #35 := Linear([[2, 2, 2, 2, 2, 2, 2]], [[0]]) #34
    #36 := Transpose #35
    X(V) := Id #36
  atom controls PostV awaits x
  init
    #17 := Transpose X(x)
    #18 := Linear([[0], [1], [2], [1], [-1], [1], [1]], [[-0], [-1], [0], [-2], [-1], [-1], [1]]) #17
    #19 := Transpose #18
    #20 := ReLU #19
    #21 := Transpose #20
    #22 := Linear([[2, 2, 2, 2, 2, 2,

## Verification

`x'` comes from the program's `update`; `V` is evaluated on `x` and `x'` (cast to real).
The domain is the loop condition `x >= 1`.

In [ ]:
#xv = list(program.ctrl)[0]
from zrth import Bool, Real, LRA, LIA, Combinatorial
import numpy as np
from zrth.z3 import interpret, fresh

enc = {}
enc.update({x: np.array([[z3.Real("x")]], dtype=object), 
              X(x0): np.array([[z3.Real("X(x0)")]], dtype=object)}) # from reads and awaits above

solver = z3.Solver()
solver.add((s > 1 for s in enc[x].flat)) # assumption (could use a module with partial update?)


for atom in M.atoms:
    for term in atom.update:
        assert len(term.write) == 1
        wire = term.write[0]
        val = interpret(term.itype)(*[enc[w] for w in term.read])
        sym = fresh(wire.dtype, f"#{wire.id}")
        enc[wire] = sym
        assert sym.shape == val.shape
        solver.add((s == v for (s,v) in zip(sym.flat, val.flat)))



**Condition 1** — `V >= 0`.

In [ ]:
solver.push()
solver.add(enc[X(V)].item() < 0) # negate
if solver.check() == z3.unsat:
    print("VERIFIED: PreV >= 0 on the domain")
else:
    print("COUNTEREXAMPLE:", solver.model())
solver.pop()

VERIFIED: PreV >= 0 on the domain


**Condition 2** — `V - PostV >= 1`.

In [ ]:
solver.push()
solver.add(enc[X(V)].item() - enc[X(PostV)].item() < 1.)         # negate
if solver.check() == z3.unsat:
    print(f"VERIFIED: V - PostV >= 1. at every step on the domain")
else:
    m = solver.model()
    print(f"COUNTEREXAMPLE: x = {m[enc[x].item()]}, X(x) = {m[enc[X(x)].item()]}, V = {m[enc[X(V)].item()]}, PostV = {m[enc[X(PostV)].item()]}")
solver.pop()

VERIFIED: V - PostV >= 1. at every step on the domain


## Result

Both conditions hold on the domain, so `V` is a ranking function and the loop terminates.